# Long-Decode Compression Benchmark (LDCB)

This notebook provides a convenient interface to run and analyze the LDCB benchmarking suite. 
LDCB isolates **decoding-time** compression behavior to compare APKVC against baselines like KIVI-int2 and KIVI-int4.

## 1. Setup
Ensure you are in the root directory of the `CommonKV` repository.

In [ ]:
import os
import sys
import json
import glob
import torch
from IPython.display import Image, display

# Add the current directory to sys.path to import ldcb modules
sys.path.append(os.getcwd())

## 2. Run Benchmarks

You can run the full benchmark suite using the command line script through the notebook. 
Adjust the `--model_id` and `--tasks` as needed.

In [ ]:
# Example: Run all tasks on Llama-2-7b-hf
!python3 -m ldcb.run_benchmark --model_id meta-llama/Llama-2-7b-hf --tasks continuation,reasoning,multiturn

## 3. Visualization

The benchmark script automatically generates plots in `ldcb/results/`. 
You can view them here by finding the latest generated files.

In [ ]:
results_dir = "ldcb/results"
plots = glob.glob(os.path.join(results_dir, "*.png"))
plots.sort(key=os.path.getmtime, reverse=True)

if plots:
    print(f"Displaying latest plot: {plots[0]}")
    display(Image(filename=plots[0]))
else:
    print("No plots found in results directory.")

## 4. Custom APKVC Ablation Sweep

Use this cell to run a manual ablation sweep as described in the LDCB guide.

In [ ]:
from ldcb.methods.apkvc import APKVCMethod
from ldcb.tasks.continuation import run_continuation
from ldcb.run_benchmark import load_model

# Load model once
model_id = "meta-llama/Llama-2-7b-hf"
model, tokenizer = load_model(model_id)

ablation_configs = [
    {"predictor_type": "none", "name": "APKVC-anchor-only"},
    {"predictor_type": "identity", "name": "APKVC-identity"},
    {"predictor_type": "linear", "name": "APKVC-linear"},
]

ablation_results = {}
for cfg in ablation_configs:
    print(f"\nRunning ablation: {cfg['name']}")
    method = APKVCMethod(predictor_type=cfg['predictor_type'])
    ablation_results[cfg['name']] = run_continuation(method, model, tokenizer)

print("\nAblation Summary:")
print(f"{'Method':<20} {'Compression':>12} {'Perplexity':>12}")
for name, r in ablation_results.items():
    cr = r.get('final_compression_ratio', {}).get('mean', 0)
    pp = r.get('perplexity', {}).get('mean', 0)
    print(f"{name:<20} {cr:>12.3f} {pp:>12.2f}")